<h3> Importar Paquetes </h3>

En este punto de la materia, seguro estás muy familiarizad@ ya con la mayoría de estos paquetes. Si alguno no te suena, o te causa intriga, pide a ChatGPT que te explique para qué sirve.

In [3]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC

from bs4 import BeautifulSoup
import re

sns.set_style('darkgrid')

<h3>Exploración </h3>

Comencemos inspeccionando los datos:

In [5]:
df = pd.read_csv('./data/IMDB Dataset.csv')


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


Ahora veamos algunas observaciones al azar:

In [7]:
df.sample(5)

,review,sentiment
33108,May contain spoilers<br /><br />This historica...,positive
11268,Answer: despite that fact that this film was w...,positive
158,The story is very trustworthy and powerful. Th...,negative
31468,You should not take what I am about to say lig...,positive
8095,"Alright, the first time I seen ""Talk Radio"" wa...",positive


Nuestro conjunto de datos tiene dos columnas:

Review:  Reseña que escribió alguna persona acerca de una película (no sabemos a qué película fue, pero no es relevante para este análisis).

Sentiment: Si la reseña fue positiva o negativa (negative o positive)

Finalmente, veamos cuántas reseñas positivas y cuántas negativas tenemos:

In [9]:
df.sentiment.value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

<h3> 25,000 reseñas positivas y 25,000 negativas </h3>

<h3> Reflexión inicial </h3>

Estamos por comenzar a hacer análisis de sentimiento usando Python. Nuestro modelo será capaz de determinar si una reseña es positiva o negativa. Por ejemplo, después de entrenar nuestro modelo, si le pedimos que analice el texto “I really loved this movie,” nuestro modelo debería indicarnos que se trata de una reseña positiva.

Esto podría dar la impresión de que la computadora está comprendiendo el texto o el lenguaje de la misma manera que lo haría un ser humano. Sin embargo, lo que haremos es algo que, aunque suene sorprendente, es fundamental en el procesamiento de lenguaje natural. Vamos a convertir los enunciados en una representación numérica. A esto se le conoce como vectorización del texto.

Mediante técnicas como TF-IDF, podemos transformar las palabras y frases en vectores, que son simplemente listas de números que representan la importancia de cada término en el contexto del texto. Estos vectores son los que el modelo utiliza para "comprender" el contenido y realizar predicciones sobre el sentimiento de la reseña.

Este proceso no implica comprensión del lenguaje en el sentido humano, sino una interpretación matemática que permite al modelo hacer inferencias basadas en patrones observados durante su entrenamiento.

<h3> Preparación de datos </h3>

Este proceso de entrenamiento será mucho más pesado que otros que hemos visto anteriormente. Por lo tanto, para fines demostrativos, reduciremos el tamaño de nuestros datos. Tomaremos 5,000 reseñas positivas y 5,000 negativas

In [10]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

Ahora procedemos a hacer nuestro split de datos de entrenamiento y de pruebas;

In [11]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

Si inspeccionamos train_y, vemos

In [12]:
train_y.value_counts()

sentiment
negative    3378
positive    3322
Name: count, dtype: int64

<h3> De texto a números </h3>

Ahora procederemos a transformar nuestros valores de texto a valores numéricos. Para esto, usaremos Scikit Learn. Específicamente, utilizaremos la clase TfidfVectorizer para transformar texto en una representación numérica utilizando el esquema TF-IDF (Term Frequency-Inverse Document Frequency).

Nuestro código es el siguiente:

In [13]:
tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)

In [ ]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [ ]:
train_y.value_counts()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

train_x_vector = tfidf.fit_transform(train_x)

In [ ]:
train_x.shape

In [ ]:
train_x_vector.shape

In [ ]:
type(train_x_vector)

In [ ]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [ ]:
primera_resenia

In [ ]:
train_x.iloc[0]

In [ ]:
primera_resenia[primera_resenia != 0]

In [ ]:
train_x.iloc[0]

In [ ]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

In [ ]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

In [ ]:
print(svc.score(test_x_vector, test_y))

In [ ]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

In [ ]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat